# Phase 1 — Tabular data: multivariate structure (Solution)

**Mục tiêu:** khi bảng có *nhiều cột số cùng lúc*, dùng small multiples / pair views / parallel coordinates để thấy cấu trúc, không nhồi hết vào một scatter.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data" / "gapminder.csv").exists():
            return p
    raise FileNotFoundError(
        "Cannot locate data/gapminder.csv from current working directory"
    )

root = resolve_repo_root()


sns.set_theme(style="whitegrid")
df = pd.read_csv(root / "data" / "gapminder.csv")
d2007 = df[df["year"] == 2007].copy()
d2007.head()


## 1) Pairplot (subset cột — tránh O(n²) quá lớn)


In [ ]:
cols = ["lifeExp", "gdpPercap", "pop"]
g = sns.pairplot(
    d2007[cols + ["continent"]],
    hue="continent",
    corner=True,
    plot_kws={"alpha": 0.6, "s": 25},
)
g.fig.suptitle("Pairplot (lifeExp, gdpPercap, pop)", y=1.02)
plt.show()


## 2) FacetGrid: cùng quan hệ, tách theo continent


In [ ]:
g = sns.FacetGrid(d2007, col="continent", col_wrap=3, height=3, sharex=False, sharey=True)
g.map_dataframe(sns.scatterplot, x="gdpPercap", y="lifeExp", alpha=0.7)
for ax in g.axes.flatten():
    ax.set_xscale("log")
g.fig.suptitle("lifeExp vs gdpPercap — small multiples", y=1.02)
plt.tight_layout()
plt.show()


## 3) Parallel coordinates (Plotly) — đọc nhiều chiều trên một canvas


In [ ]:
pc = d2007.assign(log_pop=np.log10(np.maximum(d2007["pop"].to_numpy(), 1.0))).copy()
fig = px.parallel_coordinates(
    pc,
    dimensions=["lifeExp", "gdpPercap", "log_pop"],
    color="lifeExp",
    title="Parallel coordinates (chuẩn hoá trục trong UI)",
)
fig.show()


## Reflection

- Pairplot dễ overload: bạn chọn subset cột theo tiêu chí nào?
- Parallel coordinates: đọc “giao cắt đường” cần tránh over-interpret gì?
- Small multiples vs một scatter chung: trade-off về cognitive load?


## Legacy add-on (tach tu `legacy_chapter_aligned/solution.executed.ipynb`)

Phan bo sung nay duoc ghep lai tu notebook cu de giu noi dung theo giao trinh, nhung da map vao track tabular hien tai.

### Imported section: ## 1) Coordinate systems and axes
Mục tiêu: so sánh cùng dữ liệu trên hệ trục rectilinear và radial để thấy trade-off.

## 1) Coordinate systems and axes
Mục tiêu: so sánh cùng dữ liệu trên hệ trục rectilinear và radial để thấy trade-off.

In [ ]:
d = df[df["year"] == 2007].groupby("continent", as_index=False).agg(pop=("pop", "sum"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(d["continent"], d["pop"])
axes[0].set_title("Rectilinear bar")
axes[0].tick_params(axis="x", rotation=20)

angles = np.linspace(0, 2*np.pi, len(d), endpoint=False)
r = d["pop"].to_numpy()
ax = plt.subplot(1,2,2, projection="polar")
ax.bar(angles, r, width=2*np.pi/len(d), alpha=0.7)
ax.set_title("Radial bar")
plt.tight_layout()
plt.show()

### Imported section: ## 6) Visualizing proportions
Mục tiêu: pie vs normalized stacked bar.

## 6) Visualizing proportions
Mục tiêu: pie vs normalized stacked bar.

In [ ]:
prop = d2007.groupby("continent", as_index=False).agg(pop=("pop","sum"))
prop["pct"] = prop["pop"]/prop["pop"].sum()

fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].pie(prop["pct"], labels=prop["continent"], autopct="%.1f%%")
axes[0].set_title("Pie chart")

cum = 0
for _, row in prop.sort_values("pct", ascending=False).iterrows():
    axes[1].barh(["All"], [row["pct"]], left=[cum], label=row["continent"])
    cum += row["pct"]
axes[1].set_xlim(0,1)
axes[1].set_title("Normalized stacked bar")
axes[1].legend(bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout(); plt.show()